## SALARY SENSE PROJECT!!

# "SALARY PREDICTOR" Capstone Project

# Goal

Train a fine-tuned LLM that can estimate the expected salary of a job posting.

# Order of play

1. Step 1 — Dataset Exploration
2. Step 2 — Data Curation
3. Step 3 — LLM Extraction / Preprocessing
4. Step 4 — Prompt Generation
5. Step 5 — Dataset Preparation (JSONL)
6. Step 6 — Fine-Tuning Frontier Model
7. Step 7 — Evaluation
8. Step 8 — Model Comparison



Today we'll scrub our dataset from hugginface

The dataset is here:  
https://huggingface.co/datasets/princekhunt19/2025-ai-data-jobs-dataset

And the folder with all the product datasets is here:  
https://huggingface.co/datasets/princekhunt19/2025-ai-data-jobs-dataset/tree/main/data

# Step 7 — Evaluation

## Load dataset and rebuild items

In [ ]:
import json
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.job_item import JobItem

with open("../data/fine_tune/test_set.json") as f:
    data = json.load(f)

items = [JobItem(**row) for row in data]

for item in items:
    item.make_prompt()

print("Evaluation samples:", len(items))

## Run Prediction

In [ ]:
from openai import OpenAI
import numpy as np
import re

client = OpenAI()

MODEL = "ft:gpt-3.5-turbo-0125:personal:salarysense:DG1gFFGr"

true = []
pred = []

for item in items:

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"user","content":item.prompt}],
        temperature=0
    )

    text = response.choices[0].message.content

    # Extract numeric salary safely
    number = re.search(r"\d+", text.replace(",", ""))
    predicted_salary = float(number.group())

    true.append(item.salary)
    pred.append(predicted_salary)

print("Predictions collected:", len(pred))

## Compute Error

In [ ]:
mae = np.mean(np.abs(np.array(true) - np.array(pred)))

print("Fine-tuned model MAE:", mae)